# Figure S19

Draws additional representative-site AEM resistivity sections and their locations.


In [ ]:
from pathlib import Path
import json
import shutil
import subprocess

import geopandas as gpd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import numpy as np
import pandas as pd
import rasterio
from shapely.geometry import Point


def find_repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'notebooks' / 'Fig4_resistivity_sections.jl').exists():
            return candidate
    raise FileNotFoundError('Could not find the project root.')


def display_path(path: Path) -> str:
    return str(path.resolve().relative_to(ROOT))


ROOT = find_repo_root()
NOTEBOOK_DIR = ROOT / 'notebooks'
JULIA_SCRIPT = NOTEBOOK_DIR / 'Fig4_resistivity_sections.jl'
TIF_PATH = ROOT / 'data' / '1 resistivity' / 'aem_log10res_1km_masked.tif'
DEPTH_PATH = ROOT / 'data' / '1 resistivity' / 'aem_depth_levels_m.json'
LOOKUP_PATH = ROOT / 'data' / '2 well WTD' / 'active_cell_lookup.csv'
IDOMAIN_PATH = ROOT / 'data' / '1 resistivity' / 'idomain_1km_target.dat'
TOPO_PATH = ROOT / 'data' / '5 topography' / 'topo_1km.csv'
LABELS_PATH = ROOT / 'outputs' / 'RECON_MAIN_2011_2023' / 'metrics' / 'clustering' / 'cluster_labels.csv'
WELL_LOOKUP_PATH = ROOT / 'data' / '2 well WTD' / '2011_2023' / 'well_lookup.csv'
OBS_PATH = ROOT / 'data' / '2 well WTD' / '2011_2023' / 'wtd_monthly.csv'
BOUNDARY_PATH = ROOT / 'data' / '0 mask MRVA' / 'outerboundary.shp'
OUT_DIR = ROOT / 'outputs' / 'figures' / 'FigS19'
SOURCE_DIR = OUT_DIR / 'source_data'
TARGETS_CSV = SOURCE_DIR / 'figs19_representative_sites.csv'
FIG3_SITES_PATH = ROOT / 'outputs' / 'figures' / 'Fig3' / 'source_data' / 'fig3_selected_sites.csv'
MAP_PATH = OUT_DIR / 'FigS19_representative_sites_map.png'
OUT_DIR.mkdir(parents=True, exist_ok=True)
SOURCE_DIR.mkdir(parents=True, exist_ok=True)

HALF_WINDOW_KM = 20
COLOR_RANGE = (0.3, 2.0)
TARGET_CONFIG = [
    {'set_id': '1', 'plot_order': 1, 'grid_id': 27493, 'response_class': 'Fast recovery', 'plot_label': 'Sunflower'},
    {'set_id': '1', 'plot_order': 2, 'grid_id': 79696, 'response_class': 'Slow recovery', 'plot_label': 'Malden'},
    {'set_id': '1', 'plot_order': 3, 'grid_id': 73449, 'response_class': 'Buffered', 'plot_label': 'Steele'},
    {'set_id': '2', 'plot_order': 1, 'grid_id': 14215, 'response_class': 'Fast recovery', 'plot_label': 'WC-230'},
    {'set_id': '2', 'plot_order': 2, 'grid_id': 31948, 'response_class': 'Slow recovery', 'plot_label': 'Merigold'},
    {'set_id': '2', 'plot_order': 3, 'grid_id': 47494, 'response_class': 'Buffered', 'plot_label': '01S06W12BAB1'},
]
OUTPUT_PREFIXES = {'1': 'FigS19_set1', '2': 'FigS19_set2'}
EXPORT_DPI = 600

print('Julia script:', display_path(JULIA_SCRIPT))
print('Targets table:', display_path(TARGETS_CSV))
print('Output directory:', display_path(OUT_DIR))

In [ ]:
def find_julia() -> str | None:
    candidates = []
    found = shutil.which('julia')
    if found:
        candidates.append(Path(found))
    for base in [
        Path.home() / 'AppData' / 'Local' / 'Programs',
        Path.home() / 'AppData' / 'Local' / 'Microsoft' / 'WinGet' / 'Packages',
        Path('C:/Program Files'),
    ]:
        if base.exists():
            candidates.extend(base.rglob('julia.exe'))
    for candidate in candidates:
        if candidate.exists():
            return str(candidate)
    return None


def run_cmd(args, *, check=True):
    shown = ['julia' if idx == 0 else str(value).replace(str(ROOT), '.') for idx, value in enumerate(args)]
    print('> ' + ' '.join(shown))
    proc = subprocess.run(
        [str(value) for value in args], cwd=ROOT, text=True, encoding='utf-8',
        errors='replace', stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    )
    print(proc.stdout.replace(str(ROOT), '.'))
    if check and proc.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {proc.returncode}')
    return proc


julia = find_julia()
if julia is None:
    raise RuntimeError('Julia was not found. Install Julia and rerun this notebook.')
run_cmd([julia, '--version'])

In [ ]:
config = pd.DataFrame(TARGET_CONFIG)
labels = pd.read_csv(LABELS_PATH)
wells = pd.read_csv(WELL_LOOKUP_PATH, low_memory=False)
observations = pd.read_csv(OBS_PATH, usecols=['node_id', 'month_label'])

representative = wells[wells['is_representative_site'].astype(str).str.lower().eq('true')].copy()
representative = representative.sort_values(
    ['node_id', 'n_months_observed', 'n_site_days'], ascending=[True, False, False]
).drop_duplicates('node_id')

observations['month_label'] = observations['month_label'].astype(str)
observation_summary = observations.groupby('node_id').agg(
    observed_months=('month_label', 'nunique'),
    covered_years=('month_label', lambda values: values.str[:4].nunique()),
    obs_2012_event=('month_label', lambda values: values.between('2012-01', '2013-04').sum()),
).reset_index()

sites = config.merge(
    labels[['grid_id', 'row', 'col', 'x', 'y', 'response_class', 'best_score', 'score_margin']],
    on='grid_id', how='left', suffixes=('_expected', ''),
)
sites = sites.merge(
    representative[['node_id', 'station_nm', 'lon', 'lat', 'well_depth_va', 'aqfr_cd', 'nat_aqfr_cd', 'n_months_observed']],
    left_on='grid_id', right_on='node_id', how='left',
).merge(observation_summary, on='node_id', how='left')

if sites['grid_id'].isna().any() or sites['station_nm'].isna().any():
    raise ValueError('One or more configured sites are missing from the current inputs.')
if not sites['response_class'].eq(sites['response_class_expected']).all():
    raise ValueError('A configured representative site no longer matches its expected response class.')
if not (sites.groupby('response_class').size().reindex(['Fast recovery', 'Slow recovery', 'Buffered']) == 2).all():
    raise ValueError('The selection must contain exactly two sites per response class.')

boundary = gpd.read_file(BOUNDARY_PATH).to_crs('EPSG:5070').geometry.union_all().boundary
sites['edge_distance_km'] = [Point(x, y).distance(boundary) / 1000 for x, y in zip(sites['x'], sites['y'])]

depths_m = np.asarray(json.loads(DEPTH_PATH.read_text(encoding='utf-8')), dtype=float)[:40]
band_masks = {
    'rho_0_50': (depths_m >= 0) & (depths_m < 50),
    'rho_50_120': (depths_m >= 50) & (depths_m < 120),
    'rho_120_200': (depths_m >= 120) & (depths_m < 200),
}

with rasterio.open(TIF_PATH) as src:
    stack = src.read(indexes=range(1, 41)).astype(np.float32)
    if src.nodata is not None:
        stack[stack == src.nodata] = np.nan
    ew_valid, ns_valid = [], []
    band_values, high_res_thickness, concept_matches, concept_rules = [], [], [], []
    for row in sites.itertuples():
        raster_row = src.height - 1 - int(row.row)
        col = int(row.col)
        ew = stack[:40, raster_row, col - HALF_WINDOW_KM:col + HALF_WINDOW_KM + 1]
        ns = stack[:40, raster_row - HALF_WINDOW_KM:raster_row + HALF_WINDOW_KM + 1, col]
        ew_valid.append(float(np.isfinite(ew).mean()))
        ns_valid.append(float(np.isfinite(ns).mean()))

        local = stack[:, raster_row - 5:raster_row + 6, col - 5:col + 6]
        profile = np.nanmedian(local, axis=(1, 2))
        rho = {name: float(np.nanmedian(profile[mask])) for name, mask in band_masks.items()}
        band_values.append(rho)
        high_thickness = float(np.count_nonzero(profile >= np.log10(20.0)) * 5.0)
        high_res_thickness.append(high_thickness)

        fast_contrast = rho['rho_0_50'] - 0.5 * (rho['rho_50_120'] + rho['rho_120_200'])
        buffered_contrast = min(
            rho['rho_0_50'] - rho['rho_50_120'],
            rho['rho_120_200'] - rho['rho_50_120'],
        )
        if row.response_class == 'Fast recovery':
            concept_matches.append(fast_contrast >= 0.40)
            concept_rules.append('shallow-minus-lower contrast >= 0.40 log10 ohm m')
        elif row.response_class == 'Slow recovery':
            concept_matches.append(high_thickness >= 100.0)
            concept_rules.append('local thickness above 20 ohm m >= 100 m')
        else:
            concept_matches.append(buffered_contrast >= 0.20)
            concept_rules.append('shallow and deep exceed middle by >= 0.20 log10 ohm m')
sites['aem_ew_valid_fraction'] = ew_valid
sites['aem_ns_valid_fraction'] = ns_valid
sites = pd.concat([sites.reset_index(drop=True), pd.DataFrame(band_values)], axis=1)
for band in ['0_50', '50_120', '120_200']:
    sites[f'median_rho_{band}_ohm_m'] = 10.0 ** sites[f'rho_{band}']
sites['high_resistivity_thickness_m'] = high_res_thickness
sites['concept_structure_match'] = concept_matches
sites['concept_structure_rule'] = concept_rules

if (sites['edge_distance_km'] < 20).any():
    raise ValueError('A configured site is less than 20 km from the MRVA boundary.')
if (sites['observed_months'] < 80).any() or (sites['covered_years'] < 10).any():
    raise ValueError('A configured site does not meet the observation-continuity requirement.')
if (sites[['aem_ew_valid_fraction', 'aem_ns_valid_fraction']].min(axis=1) < 0.98).any():
    raise ValueError('A configured site has insufficient AEM coverage in its 40-km sections.')
if not sites['concept_structure_match'].all():
    failed = sites.loc[~sites['concept_structure_match'], ['grid_id', 'response_class', 'concept_structure_rule']]
    raise ValueError('A configured site does not match its concept structure:\n' + failed.to_string(index=False))

sites = sites.sort_values(['set_id', 'plot_order']).reset_index(drop=True)
output_columns = [
    'set_id', 'plot_order', 'grid_id', 'response_class', 'plot_label', 'row', 'col', 'x', 'y',
    'lon', 'lat', 'station_nm', 'best_score', 'score_margin', 'well_depth_va', 'aqfr_cd',
    'nat_aqfr_cd', 'observed_months', 'covered_years', 'obs_2012_event', 'edge_distance_km',
    'aem_ew_valid_fraction', 'aem_ns_valid_fraction', 'median_rho_0_50_ohm_m',
    'median_rho_50_120_ohm_m', 'median_rho_120_200_ohm_m',
    'high_resistivity_thickness_m', 'concept_structure_match',
    'concept_structure_rule',
]
sites[output_columns].to_csv(TARGETS_CSV, index=False, lineterminator='\n')
print(sites[[
    'set_id', 'grid_id', 'response_class', 'plot_label', 'station_nm',
    'observed_months', 'covered_years', 'obs_2012_event', 'edge_distance_km',
    'aem_ew_valid_fraction', 'aem_ns_valid_fraction', 'median_rho_0_50_ohm_m',
    'median_rho_50_120_ohm_m', 'median_rho_120_200_ohm_m',
    'high_resistivity_thickness_m', 'concept_structure_match',
]].to_string(index=False))
print('Saved:', display_path(TARGETS_CSV))

In [ ]:
for set_id, output_prefix in OUTPUT_PREFIXES.items():
    run_cmd([
        julia,
        f'--project={NOTEBOOK_DIR}',
        JULIA_SCRIPT,
        '--output', OUT_DIR,
        '--output-prefix', output_prefix,
        '--targets-csv', TARGETS_CSV,
        '--target-set', set_id,
        '--half-window-km', str(HALF_WINDOW_KM),
        '--color-range', ','.join(map(str, COLOR_RANGE)),
        '--png-only',
        '--tif', TIF_PATH,
        '--depths', DEPTH_PATH,
        '--lookup', LOOKUP_PATH,
        '--idomain', IDOMAIN_PATH,
        '--topo', TOPO_PATH,
    ])

In [ ]:
from IPython.display import Image, display

for output_prefix in OUTPUT_PREFIXES.values():
    path = OUT_DIR / f'{output_prefix}_resistivity_cutaway_3D.png'
    if not path.exists():
        raise FileNotFoundError(path)
    print(display_path(path))
    display(Image(filename=str(path)))

CLASS_ORDER = ['Fast recovery', 'Slow recovery', 'Buffered']
CLASS_COLORS = {
    'Fast recovery': '#2D5FB8',
    'Slow recovery': '#C44E72',
    'Buffered': '#13A8A2',
}
mpl.rcParams.update({
    'font.family': 'Arial',
    'font.size': 9,
    'axes.linewidth': 0.65,
    'axes.unicode_minus': False,
})
main_sites = pd.read_csv(FIG3_SITES_PATH)
extra_sites = sites.copy()
boundary_map = gpd.read_file(BOUNDARY_PATH).to_crs('EPSG:4326')
fig, ax = plt.subplots(figsize=(3.35, 5.8), dpi=EXPORT_DPI)
boundary_map.plot(ax=ax, facecolor='#F3F4F1', edgecolor='#1F2933', linewidth=1.25, zorder=1)
for class_name in CLASS_ORDER:
    color = CLASS_COLORS[class_name]
    main = main_sites[main_sites['response_class'].eq(class_name)]
    extra = extra_sites[extra_sites['response_class'].eq(class_name)]
    ax.scatter(main['lon'], main['lat'], s=170, marker='*', facecolor=color, edgecolor='black', linewidth=0.8, zorder=5)
    ax.scatter(extra['lon'], extra['lat'], s=68, marker='o', facecolor=color, edgecolor='white', linewidth=0.9, zorder=4)
for row in extra_sites.itertuples():
    ax.annotate(str(row.set_id), (row.lon, row.lat), xytext=(5, 0), textcoords='offset points', ha='left', va='center', fontsize=8, fontweight='bold', color='#111827', zorder=6)
minx, miny, maxx, maxy = boundary_map.total_bounds
ax.set_xlim(minx - 0.10, maxx + 0.18)
ax.set_ylim(miny - 0.10, maxy + 0.10)
ax.set_xticks([])
ax.set_yticks([])
ax.tick_params(axis='both', which='both', bottom=False, left=False,
               labelbottom=False, labelleft=False)
mean_lat = 0.5 * (miny + maxy)
ax.set_aspect(1.0 / np.cos(np.deg2rad(mean_lat)), adjustable='box')
ax.grid(False)
for spine in ax.spines.values():
    spine.set_visible(False)
handles = [Patch(facecolor=CLASS_COLORS[name], edgecolor='none', label=name) for name in CLASS_ORDER]
handles += [
    Line2D([0], [0], marker='*', color='none', markerfacecolor='white', markeredgecolor='black', markersize=13, label='Sites in Fig. 3'),
]
ax.legend(handles=handles, loc='lower right', frameon=True, framealpha=0.92, edgecolor='#B8BEC7', fontsize=8, handlelength=1.1, borderpad=0.55)
fig.tight_layout(pad=0.8)
fig.savefig(MAP_PATH, dpi=600, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved:', display_path(MAP_PATH))